In [1]:
from collections import defaultdict
from dataclasses import dataclass
import os
import random
import time
from typing import Optional

import tqdm

from mani_skill.utils import gym_utils
from mani_skill.utils.wrappers.flatten import FlattenActionSpaceWrapper, FlattenRGBDObservationWrapper
from mani_skill.utils.wrappers.record import RecordEpisode
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import tyro

import mani_skill.envs

/opt/conda/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PlainConv(nn.Module):
    def __init__(self,
                 in_channels=3,
                 out_dim=256,
                 pool_feature_map=False,
                 last_act=True, # True for ConvBody, False for CNN
                 image_size=[128, 128]
                 ):
        super().__init__()
        # assume input image size is 128x128 or 64x64

        self.out_dim = out_dim
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4) if image_size[0] == 128 and image_size[1] == 128 else nn.MaxPool2d(2, 2),  # [32, 32]
            nn.Conv2d(16, 32, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # [16, 16]
            nn.Conv2d(32, 64, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # [8, 8]
            nn.Conv2d(64, 64, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # [4, 4]
            nn.Conv2d(64, 64, 1, padding=0, bias=True), nn.ReLU(inplace=True),
        )

        if pool_feature_map:
            self.pool = nn.AdaptiveMaxPool2d((1, 1))
            self.fc = make_mlp(128, [out_dim], last_act=last_act)
        else:
            self.pool = None
            self.fc = make_mlp(64 * 4 * 4, [out_dim], last_act=last_act)

        self.reset_parameters()

    def reset_parameters(self):
        for name, module in self.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, image):
        x = self.cnn(image)
        if self.pool is not None:
            x = self.pool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

class EncoderObsWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, obs):
        if "rgb" in obs:
            rgb = obs['rgb'].float() / 255.0 # (B, H, W, 3*k)
        if "depth" in obs:
            depth = obs['depth'].float() # (B, H, W, 1*k)
        if "rgb" and "depth" in obs:
            img = torch.cat([rgb, depth], dim=3) # (B, H, W, C)
        elif "rgb" in obs:
            img = rgb
        elif "depth" in obs:
            img = depth
        else:
            raise ValueError(f"Observation dict must contain 'rgb' or 'depth'")
        img = img.permute(0, 3, 1, 2) # (B, C, H, W)
        return self.encoder(img)

def make_mlp(in_channels, mlp_channels, act_builder=nn.ReLU, last_act=True):
    c_in = in_channels
    module_list = []
    for idx, c_out in enumerate(mlp_channels):
        module_list.append(nn.Linear(c_in, c_out))
        if last_act or idx < len(mlp_channels) - 1:
            module_list.append(act_builder())
        c_in = c_out
    return nn.Sequential(*module_list)

class SoftQNetwork(nn.Module):
    def __init__(self, envs, encoder: EncoderObsWrapper):
        super().__init__()
        self.encoder = encoder
        action_dim = np.prod(envs.single_action_space.shape)
        state_dim = envs.single_observation_space['state'].shape[0]
        self.mlp = make_mlp(encoder.encoder.out_dim+action_dim+state_dim, [512, 256, 1], last_act=False)

    def forward(self, obs, action, visual_feature=None, detach_encoder=False):
        if visual_feature is None:
            visual_feature = self.encoder(obs)
        if detach_encoder:
            visual_feature = visual_feature.detach()
        x = torch.cat([visual_feature, obs["state"], action], dim=1)
        return self.mlp(x)


LOG_STD_MAX = 2
LOG_STD_MIN = -5

class Actor(nn.Module):
    def __init__(self, envs, sample_obs):
        super().__init__()
        action_dim = np.prod(envs.single_action_space.shape)
        state_dim = envs.single_observation_space['state'].shape[0]
        # count number of channels and image size
        in_channels = 0
        if "rgb" in sample_obs:
            in_channels += sample_obs["rgb"].shape[-1]
            image_size = sample_obs["rgb"].shape[1:3]
        if "depth" in sample_obs:
            in_channels += sample_obs["depth"].shape[-1]
            image_size = sample_obs["depth"].shape[1:3]

        self.encoder = EncoderObsWrapper(
            PlainConv(in_channels=in_channels, out_dim=256, image_size=image_size) # assume image is 64x64
        )
        self.mlp = make_mlp(self.encoder.encoder.out_dim+state_dim, [512, 256], last_act=True)
        self.fc_mean = nn.Linear(256, action_dim)
        self.fc_logstd = nn.Linear(256, action_dim)
        # action rescaling
        self.action_scale = torch.FloatTensor((envs.single_action_space.high - envs.single_action_space.low) / 2.0)
        self.action_bias = torch.FloatTensor((envs.single_action_space.high + envs.single_action_space.low) / 2.0)

    def get_feature(self, obs, detach_encoder=False):
        visual_feature = self.encoder(obs)
        if detach_encoder:
            visual_feature = visual_feature.detach()
        x = torch.cat([visual_feature, obs['state']], dim=1)
        return self.mlp(x), visual_feature

    def forward(self, obs, detach_encoder=False):
        x, visual_feature = self.get_feature(obs, detach_encoder)
        mean = self.fc_mean(x)
        log_std = self.fc_logstd(x)
        log_std = torch.tanh(log_std)
        log_std = LOG_STD_MIN + 0.5 * (LOG_STD_MAX - LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std, visual_feature

    def get_eval_action(self, obs):
        mean, log_std, _ = self(obs)
        action = torch.tanh(mean) * self.action_scale + self.action_bias
        return action

    def get_action(self, obs, detach_encoder=False):
        mean, log_std, visual_feature = self(obs, detach_encoder)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # for reparameterization trick (mean + std * N(0,1))
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        log_prob = normal.log_prob(x_t)
        # Enforcing Action Bound
        log_prob -= torch.log(self.action_scale * (1 - y_t.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, mean, visual_feature

    def to(self, device):
        self.action_scale = self.action_scale.to(device)
        self.action_bias = self.action_bias.to(device)
        return super().to(device)

In [3]:
env_kwargs = dict(obs_mode='rgb', render_mode='all', sim_backend="gpu", sensor_configs=dict())
env_kwargs["control_mode"] = "pd_ee_delta_pos"
env_kwargs["sensor_configs"]["width"] = 64
env_kwargs["sensor_configs"]["height"] = 64

eval_envs = gym.make('PickCube-v1', num_envs=100, reconfiguration_freq=1, human_render_camera_configs=dict(shader_pack="default"), **env_kwargs)

eval_envs = FlattenRGBDObservationWrapper(eval_envs, rgb=True, depth=False, state=True)

#eval_envs = FlattenActionSpaceWrapper(eval_envs)

eval_envs = ManiSkillVectorEnv(eval_envs, 100, ignore_terminations=True, record_metrics=True)


/opt/conda/lib/python3.9/site-packages/torch/random.py:187: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


In [4]:
checkpoint = '/home/mount/Maniskill_OnlineTransformers/runs/MLP_RGBD_ACCELERATOR/PickCube-v1__sac_rgbd__3__1750696131/final_ckpt.pt'
ckpt = torch.load(checkpoint)
obs, info = eval_envs.reset(seed=1)
actor = Actor(eval_envs, sample_obs=obs).to('cuda')
actor.load_state_dict(ckpt['actor'])

<All keys matched successfully>

In [39]:
class DictArray(object):
    def __init__(self, buffer_shape, element_space, data_dict=None, device=None):
        self.buffer_shape = buffer_shape
        if data_dict:
            self.data = data_dict
        else:
            assert isinstance(element_space, gym.spaces.dict.Dict)
            self.data = {}
            for k, v in element_space.items():
                if isinstance(v, gym.spaces.dict.Dict):
                    self.data[k] = DictArray(buffer_shape, v, device=device)
                else:
                    dtype = (torch.float32 if v.dtype in (np.float32, np.float64) else
                            torch.uint8 if v.dtype == np.uint8 else
                            torch.int16 if v.dtype == np.int16 else
                            torch.int32 if v.dtype == np.int32 else
                            v.dtype)
                    self.data[k] = torch.zeros(buffer_shape + v.shape, dtype=dtype, device=device)

    def keys(self):
        return self.data.keys()

    def __getitem__(self, index):
        if isinstance(index, str):
            return self.data[index]
        return {
            k: v[index] for k, v in self.data.items()
        }

    def __setitem__(self, index, value):
        if isinstance(index, str):
            self.data[index] = value
        for k, v in value.items():
            self.data[k][index] = v

    @property
    def shape(self):
        return self.buffer_shape

    def reshape(self, shape):
        t = len(self.buffer_shape)
        new_dict = {}
        for k,v in self.data.items():
            if isinstance(v, DictArray):
                new_dict[k] = v.reshape(shape)
            else:
                new_dict[k] = v.reshape(shape + v.shape[t:])
        new_buffer_shape = next(iter(new_dict.values())).shape[:len(shape)]
        return DictArray(new_buffer_shape, None, data_dict=new_dict)

@dataclass
class ReplayBufferSample:
    obs: torch.Tensor
    next_obs: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
class ReplayBuffer:
    def __init__(self, env, num_envs: int, buffer_size: int, storage_device: torch.device, sample_device: torch.device):
        self.buffer_size = buffer_size
        self.pos = 0
        self.full = False
        self.num_envs = num_envs
        self.storage_device = storage_device
        self.sample_device = sample_device
        self.per_env_buffer_size = buffer_size // num_envs
        # note 128x128x3 RGB data with replay buffer size 100_000 takes up around 4.7GB of GPU memory
        # 32 parallel envs with rendering uses up around 2.2GB of GPU memory.
        self.obs = DictArray((self.per_env_buffer_size, num_envs), env.single_observation_space, device=storage_device)
        # TODO (stao): optimize final observation storage
        self.next_obs = DictArray((self.per_env_buffer_size, num_envs), env.single_observation_space, device=storage_device)
        self.actions = torch.zeros((self.per_env_buffer_size, num_envs) + env.single_action_space.shape, device=storage_device)
        self.logprobs = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.rewards = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.dones = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.values = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        
        self.associated_r = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.Q = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.V = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)


    def add_associated_reward(self, positions, rewards):
        rewards = rewards.unsqueeze(0).repeat(len(positions), 1)   #n_e -> len(positions), n_e
        self.associated_r[[positions]] += rewards
        
    def add_qv_estimates(self, positions: list[int]):
        """
        Использует associated_r и rewards для вычисления Q(s,a) и V(s)
        """
        pos = torch.tensor(positions)  # [T]
        env_ids = torch.arange(self.num_envs)  # [N]
        ti, ei = torch.meshgrid(pos, env_ids, indexing="ij")  # [T, N]

        # [T, N]
        rewards = self.rewards[ti, ei]
        associated_r = self.associated_r[ti, ei]

        # Cumulative sum по временной оси (dim=0), но без включения текущего шага для Q
        # → Q = R - cumsum без текущего
        # → V = R - cumsum включая текущий
        cum_rewards_exclusive = torch.cumsum(rewards, dim=0) - rewards  # [T, N]
        cum_rewards_inclusive = torch.cumsum(rewards, dim=0)  # [T, N]

        q_values = associated_r - cum_rewards_exclusive
        v_values = associated_r - cum_rewards_inclusive

        self.Q[ti, ei] = q_values
        self.V[ti, ei] = v_values
    
        
    
    def add(self, obs: torch.Tensor, next_obs: torch.Tensor, action: torch.Tensor, reward: torch.Tensor, done: torch.Tensor):
        if self.storage_device == torch.device("cpu"):
            obs = {k: v.cpu() for k, v in obs.items()}
            next_obs = {k: v.cpu() for k, v in next_obs.items()}
            action = action.cpu()
            reward = reward.cpu()
            done = done.cpu()

        self.obs[self.pos] = obs
        self.next_obs[self.pos] = next_obs

        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.dones[self.pos] = done

        self.pos += 1
        if self.pos == self.per_env_buffer_size:
            self.full = True
            self.pos = 0
            return self.per_env_buffer_size 
        
        return self.pos-1
    
        
    def sample(self, batch_size: int):
        if self.full:
            batch_inds = torch.randint(0, self.per_env_buffer_size, size=(batch_size, ))
        else:
            batch_inds = torch.randint(0, self.pos, size=(batch_size, ))
        env_inds = torch.randint(0, self.num_envs, size=(batch_size, ))
        obs_sample = self.obs[batch_inds, env_inds]
        next_obs_sample = self.next_obs[batch_inds, env_inds]
        obs_sample = {k: v.to(self.sample_device) for k, v in obs_sample.items()}
        next_obs_sample = {k: v.to(self.sample_device) for k, v in next_obs_sample.items()}
        return ReplayBufferSample(
            obs=obs_sample,
            next_obs=next_obs_sample,
            actions=self.actions[batch_inds, env_inds].to(self.sample_device),
            rewards=self.rewards[batch_inds, env_inds].to(self.sample_device),
            dones=self.dones[batch_inds, env_inds].to(self.sample_device)
        )
        
    def sample_for_trans(self, positions: list):    
        positions = torch.tensor(positions)  # T
        env_ids = torch.arange(self.num_envs)  # N
        ti, ei = torch.meshgrid(positions, env_ids, indexing="ij")  # T, N

        obs_batch = self.obs[ti, ei]
        next_obs_batch = self.next_obs[ti, ei]
        actions_batch = self.actions[ti, ei]
        rewards_batch = self.rewards[ti, ei]
        dones_batch = self.dones[ti, ei]

        obs_batch = {k: v.to(self.sample_device) for k, v in obs_batch.items()}
        next_obs_batch = {k: v.to(self.sample_device) for k, v in next_obs_batch.items()}

        return ReplayBufferSample(
            obs=obs_batch,
            next_obs=next_obs_batch,
            actions=actions_batch.to(self.sample_device),
            rewards=rewards_batch.to(self.sample_device),
            dones=dones_batch.to(self.sample_device),
        )
        
    def make_sequential_dataloader(self, positions: list[int], context_len: int, batch_size: int, shuffle: bool = True):
        pos = torch.tensor(positions)
        num_steps = len(pos)
        n_envs = self.num_envs
        device = self.sample_device   
        
        time_idx, env_idx = torch.meshgrid(pos, torch.arange(n_envs), indexing="ij")
        
        obs = self.obs[time_idx, env_idx]
        next_obs = self.next_obs[time_idx, env_idx]
        actions = self.actions[time_idx, env_idx]
        rewards = self.rewards[time_idx, env_idx]
        dones = self.dones[time_idx, env_idx]
        associated_r = self.associated_r[time_idx, env_idx]
        q_vals = self.Q[time_idx, env_idx]
        v_vals = self.V[time_idx, env_idx]

        sequences = []
        for start in range(num_steps - context_len + 1):
            end = start + context_len

            # Собираем последовательности: [context_len, n_envs, ...] → [n_envs, context_len, ...]
            obs_seq = {k: v[start:end].to(device).permute(1, 0, *range(2, v.ndim)) for k, v in obs.items()}
            next_obs_seq = {k: v[start:end].to(device).permute(1, 0, *range(2, v.ndim)) for k, v in next_obs.items()}
            actions_seq = actions[start:end].to(device).permute(1, 0, *range(2, actions.ndim))
            rewards_seq = rewards[start:end].to(device).permute(1, 0)
            dones_seq = dones[start:end].to(device).permute(1, 0)
            associated_r_seq = associated_r[start:end].to(device).permute(1, 0)
            q_seq = q_vals[start:end].to(device).permute(1, 0)
            v_seq = v_vals[start:end].to(device).permute(1, 0)

            for i in range(n_envs):
                seq = (
                    {k: v[i] for k, v in obs_seq.items()},           # [context, ...]
                    {k: v[i] for k, v in next_obs_seq.items()},
                    actions_seq[i],
                    rewards_seq[i],
                    dones_seq[i],
                    associated_r_seq[i],
                    q_seq[i],
                    v_seq[i]
                )
                sequences.append(seq)

        obs_seq = {k: torch.stack([s[0][k] for s in sequences]) for k in sequences[0][0]}
        next_obs_seq = {k: torch.stack([s[1][k] for s in sequences]) for k in sequences[0][1]}
        actions_seq = torch.stack([s[2] for s in sequences])
        rewards_seq = torch.stack([s[3] for s in sequences])
        dones_seq = torch.stack([s[4] for s in sequences])
        associated_r_seq = torch.stack([s[5] for s in sequences]) 
        q_seq = torch.stack([s[6] for s in sequences])
        v_seq = torch.stack([s[7] for s in sequences])   

        dataset = TensorDataset(
            *list(obs_seq.values()),
            *list(next_obs_seq.values()),
            actions_seq,
            rewards_seq,
            dones_seq,
            associated_r_seq,
            q_seq,
            v_seq
        )
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

        return dataloader

    

# Collecting train data

In [40]:
rb = ReplayBuffer(
        env=eval_envs,
        num_envs=100,
        buffer_size=100000,
        storage_device=torch.device('cuda'),
        sample_device='cuda')

In [41]:
from collections import defaultdict
from torch.utils.data import TensorDataset, DataLoader

actor.eval()
eval_obs, _ = eval_envs.reset()
eval_metrics = defaultdict(list)
num_episodes = 0
positions = []

for _ in range(50):
    with torch.no_grad():
        eval_actions = actor.get_eval_action(eval_obs)
        eval_actions = eval_actions.detach()

        eval_next_obs, eval_rew, eval_terminations, eval_truncations, eval_infos = eval_envs.step(eval_actions)

        # Добавляем в буфер (как в первом варианте)
        real_next_obs = {k: v.clone() for k, v in eval_next_obs.items()}

        # По твоей логике bootstrap_at_done = "always"
        stop_bootstrap = torch.zeros_like(eval_terminations, dtype=torch.bool)

        pos = rb.add(eval_obs, real_next_obs, eval_actions, eval_rew, stop_bootstrap)
        positions.append(pos)

        # Обновляем obs на следующий шаг
        eval_obs = eval_next_obs

        # Сохраняем метрики
        if "final_info" in eval_infos:
            mask = eval_infos["_final_info"]
            num_episodes += mask.sum()
            for k, v in eval_infos["final_info"]["episode"].items():
                eval_metrics[k].append(v)

# Вычисляем средние значения метрик
eval_metrics_mean = {}
for k, v in eval_metrics.items():
    mean = torch.stack(v).float().mean()
    eval_metrics_mean[k] = mean
    print(f"eval/{k}", mean)



rb.add_associated_reward(positions, eval_metrics['return'][0])
rb.add_qv_estimates(positions)

eval/success_once tensor(0.9700, device='cuda:0')
eval/return tensor(41.6131, device='cuda:0')
eval/episode_len tensor(50., device='cuda:0')
eval/reward tensor(0.8323, device='cuda:0')
eval/success_at_end tensor(0.9500, device='cuda:0')


In [42]:
dataloader = rb.make_sequential_dataloader(positions=positions, context_len=5, batch_size=60, shuffle=False)

In [43]:
for batch in dataloader:
    break

print(batch[0].shape,'\n', 
      batch[1].shape, '\n',
      batch[2].shape, '\n',
      batch[3].shape, '\n',
      batch[4].shape, '\n',
      batch[5].shape, '\n',
      batch[6].shape, '\n',
      batch[7].shape, '\n',
      batch[8].shape, '\n',
      batch[9].shape, '\n',)

torch.Size([60, 5, 29]) 
 torch.Size([60, 5, 64, 64, 3]) 
 torch.Size([60, 5, 29]) 
 torch.Size([60, 5, 64, 64, 3]) 
 torch.Size([60, 5, 4]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 



# Train Trans

In [ ]:
class PlainConv(nn.Module):
    '''
    Conv Net constructor
    '''
    def __init__(self,
                 in_channels=3,
                 out_dim=227, #256,
                 pool_feature_map=False,
                 last_act=True, # True for ConvBody, False for CNN
                 image_size=[128, 128]
                 ):
        super().__init__()
        # assume input image size is 128x128 or 64x64

        self.out_dim = out_dim
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4) if image_size[0] == 128 and image_size[1] == 128 else nn.MaxPool2d(2, 2),  # [32, 32]
            nn.Conv2d(16, 32, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # [16, 16]
            nn.Conv2d(32, 64, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # [8, 8]
            nn.Conv2d(64, 64, 3, padding=1, bias=True), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # [4, 4]
            nn.Conv2d(64, 64, 1, padding=0, bias=True), nn.ReLU(inplace=True),
        )

        if pool_feature_map:
            self.pool = nn.AdaptiveMaxPool2d((1, 1))
            self.fc = make_mlp(128, [out_dim], last_act=last_act)
        else:
            self.pool = None
            self.fc = make_mlp(64 * 4 * 4, [out_dim], last_act=last_act)

        self.reset_parameters()

    def reset_parameters(self):
        for name, module in self.named_modules():
            if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, image):
        x = self.cnn(image)
        if self.pool is not None:
            x = self.pool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

class EncoderObsWrapper(nn.Module):
    '''
    Preparation module before applying CNN
    '''
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, obs):
        if "rgb" in obs:
            rgb = obs['rgb'].float() / 255.0 # (B, H, W, 3*k)
        if "depth" in obs:
            depth = obs['depth'].float() # (B, H, W, 1*k)
        if "rgb" and "depth" in obs:
            img = torch.cat([rgb, depth], dim=3) # (B, H, W, C)
        elif "rgb" in obs:
            img = rgb
        elif "depth" in obs:
            img = depth
        else:
            raise ValueError(f"Observation dict must contain 'rgb' or 'depth'")
        
        #print(img.shape)
        if len(img.shape) == 5: # we are on evaluation step
            n_e, cont, h, w, c = img.shape
            img = img.reshape(n_e*cont, h, w, c)
            img = img.permute(0, 3, 1, 2) # (B, C, H, W)
            img = self.encoder(img)
            img = img.reshape(n_e, cont, self.encoder.out_dim)
            
        else:                   # we are on train step
            bs, n_e, cont, h, w, c = img.shape
            img = img.reshape(bs*n_e*cont, h, w, c)
            img = img.permute(0, 3, 1, 2) # (B, C, H, W)
            img = self.encoder(img)
            img = img.reshape(bs, n_e, cont, self.encoder.out_dim)
        
        
        return img

def make_mlp(in_channels, mlp_channels, act_builder=nn.ReLU, last_act=True):
    c_in = in_channels
    module_list = []
    for idx, c_out in enumerate(mlp_channels):
        module_list.append(nn.Linear(c_in, c_out))
        if last_act or idx < len(mlp_channels) - 1:
            module_list.append(act_builder())
        c_in = c_out
    return nn.Sequential(*module_list)



class PositionalEncoding(nn.Module):
    def __init__(self, dim, device, min_timescale=2.0, max_timescale=1e4):
        super().__init__()
        self.device = device
        freqs = torch.arange(0, dim, min_timescale).to(self.device)
        inv_freqs = max_timescale ** (-freqs / dim)
        self.register_buffer("inv_freqs", inv_freqs)

    def forward(self, seq_len):
        seq = torch.arange(seq_len - 1, -1, -1.0).to(self.device)
        sinusoidal_inp = rearrange(seq, "n -> n ()") * rearrange(self.inv_freqs, "d -> () d")
        pos_emb = torch.cat((sinusoidal_inp.sin(), sinusoidal_inp.cos()), dim=-1)
        return pos_emb

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):

    def __init__(self, config, input_dim):
        super().__init__()
        #assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(input_dim, 3 * input_dim, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(input_dim, input_dim, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = input_dim
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
    
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.num_steps, config.num_steps))
                                        .view(1, 1, config.num_steps, config.num_steps))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y



class MLP(nn.Module):

    def __init__(self, config, input_dim):
        super().__init__()
        self.c_fc    = nn.Linear(input_dim, 4 * input_dim, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * input_dim, input_dim, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):

    def __init__(self, config, inner_size):
        super().__init__()
        
        encoder = EncoderObsWrapper
        
        
        self.ln_1 = LayerNorm(inner_size, bias=config.bias)
        self.attn = CausalSelfAttention(config, inner_size)
        self.ln_2 = LayerNorm(inner_size, bias=config.bias)
        self.mlp = MLP(config, inner_size)

        if config.use_gates:
            self.skip_fn_1 = GRUGate(inner_size, 2.0)
            self.skip_fn_2 = GRUGate(inner_size, 2.0)
        else:
            self.skip_fn_1 = lambda x, y: x + y
            self.skip_fn_2 = lambda x, y: x + y


    def forward(self, x):

        x = self.skip_fn_1(x, self.attn(self.ln_1(x)))
        x = self.skip_fn_2(x, self.mlp(self.ln_2(x)))

        
        return x

class GPT(nn.Module):

    def __init__(self, config, inner_size):
        super().__init__()

        self.config = config
        self.pos_embedding = nn.Embedding(config.max_episode_steps, inner_size)

        self.transformer_layers = nn.ModuleList([Block(config, inner_size) for _ in range(config.n_layer)])
        self.ln_f = LayerNorm(inner_size, bias=config.bias)
        self.drop = nn.Dropout(config.dropout)

        
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    

    def get_num_params(self, non_embedding=True):
        
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):

        t = x.shape[1]

        device = x.device
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)
        pos_emb = self.pos_embedding(pos) # position embeddings of shape (t, n_embd)
        
        #print(f"x: {x.shape}") 
        #print(f"pe: {pos_emb.shape}")  
        
        x = self.drop(x + pos_emb)
        for block in self.transformer_layers:
            x = block(x)
        x = self.ln_f(x)

        return x
    
    
    
class Trans_Actor(nn.Module):
    def __init__(self, envs, args, sample_obs):
        super().__init__()
        action_dim = np.prod(envs.single_action_space.shape)
        self.state_dim = envs.single_observation_space['state'].shape[0] if 'state' in envs.single_observation_space.keys() else 0
        print(f"actor sd = {self.state_dim}")
        # count number of channels and image size
        in_channels = 0
        if "rgb" in sample_obs:
            in_channels += sample_obs["rgb"].shape[-1]
            image_size = sample_obs["rgb"].shape[1:3]
        if "depth" in sample_obs:
            in_channels += sample_obs["depth"].shape[-1]
            image_size = sample_obs["depth"].shape[1:3]

        self.encoder = EncoderObsWrapper(
            PlainConv(in_channels=in_channels, out_dim=227, image_size=image_size) # assume image is 64x64
        )
        inner_size = self.encoder.encoder.out_dim+self.state_dim
        self.fc_mean = nn.Linear(inner_size, action_dim)
        self.fc_logstd = nn.Linear(inner_size, action_dim)
        self.action_scale = torch.FloatTensor((envs.single_action_space.high - envs.single_action_space.low) / 2.0)
        self.action_bias = torch.FloatTensor((envs.single_action_space.high + envs.single_action_space.low) / 2.0)
        
        self.transformer = GPT(args, inner_size)

    def get_feature(self, obs, detach_encoder=False):
        #print(f"x before cnn {obs[args.obs_mode].shape} and {obs['state'].shape}")
        visual_feature = self.encoder(obs)
        #print(f"x before cat with state {visual_feature.shape}")
        if detach_encoder:
            visual_feature = visual_feature.detach()
        x = torch.cat([visual_feature, obs['state']], dim=-1)
        #print(f"x after cat with state {x.shape}")
    
        return self.transformer(x)[:,-1,:], visual_feature
    
    def forward(self, obs, detach_encoder=False):
        x, visual_feature = self.get_feature(obs, detach_encoder)
        mean = self.fc_mean(x)
        log_std = self.fc_logstd(x)
        log_std = torch.tanh(log_std)
        log_std = LOG_STD_MIN + 0.5 * (LOG_STD_MAX - LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std, visual_feature

    def get_eval_action(self, obs):
        mean, log_std, _ = self(obs)
        self.action_scale = self.action_scale.to(mean.device)
        self.action_bias = self.action_bias.to(mean.device)
        action = torch.tanh(mean) * self.action_scale + self.action_bias
        return action

    def get_action(self, obs, detach_encoder=False):
        mean, log_std, visual_feature = self(obs, detach_encoder)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # for reparameterization trick (mean + std * N(0,1))
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        log_prob = normal.log_prob(x_t)
        # Enforcing Action Bound
        log_prob -= torch.log(self.action_scale * (1 - y_t.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, mean, visual_feature

    def to(self, device):
        self.action_scale = self.action_scale.to(device)
        self.action_bias = self.action_bias.to(device)
        return super().to(device)
    


class Trans_SoftQNetwork(nn.Module):
    '''
    Q-network for Transformer-based maniskill tasks
    '''
    def __init__(self, env, args, encoder: EncoderObsWrapper):
        super().__init__()
        self.encoder = encoder
        action_dim = np.prod(env.single_action_space.shape)
        self.state_dim = env.single_observation_space['state'].shape[0] if 'state' in env.single_observation_space.keys() else 0
        print(f"q_net sd = {self.state_dim}")
        inner_size = encoder.encoder.out_dim+self.state_dim
        
        self.transformer = GPT(args, inner_size)
        
        self.net = nn.Sequential(
            nn.Linear(encoder.encoder.out_dim+action_dim+self.state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, obs, action, visual_feature=None, detach_encoder=False):
        if visual_feature is None:
            visual_feature = self.encoder(obs) # img -> vec
        if detach_encoder:
            visual_feature = visual_feature.detach()
        if self.state_dim != 0:
            trans_inp = torch.cat([visual_feature, obs["state"]], dim=-1)
            trans_out = self.transformer(trans_inp)[:, -1, :]
        else:
            trans_out = self.transformer(visual_feature)[:, -1, :]
        x = torch.cat([trans_out, action], dim=-1) 
        
        return self.net(x)    
    
    

In [103]:
import math 
from types import SimpleNamespace

eval_envs.single_observation_space.dtype = np.float32
obs, info = eval_envs.reset()
max_episode_steps = gym_utils.find_max_episode_steps_value(eval_envs._env)

args = {
    "use_gates": False,
    "n_embd": 227,
    "n_layer": 1,
    "n_head": 2,
    "dropout": 0.0,
    "seq_len": 5,
    "bias": True,
    'max_episode_steps': max_episode_steps
}
config = SimpleNamespace(**args)


trans_actor = Trans_Actor(envs=eval_envs, args=config, sample_obs=obs).to('cuda')
trans_qf1 = Trans_SoftQNetwork(eval_envs, config, trans_actor.encoder).to('cuda')
trans_qf2 = Trans_SoftQNetwork(eval_envs, config, trans_actor.encoder).to('cuda')
    
trans_qf1_target = Trans_SoftQNetwork(eval_envs, config, actor.encoder).to('cuda')
trans_qf2_target = Trans_SoftQNetwork(eval_envs, config, actor.encoder).to('cuda')
    
q_optimizer = optim.Adam(
                    list(trans_qf1.transformer.parameters()) +
                    list(trans_qf2.transformer.parameters()) +
                    list(trans_qf1.net.parameters()) +
                    list(trans_qf2.net.parameters()) +
                    list(trans_qf1.encoder.parameters()),
                    lr=3e-4)
actor_optimizer = optim.Adam(list(actor.parameters()), lr=3e-4)
    

    # Automatic entropy tuning
target_entropy = -torch.prod(torch.Tensor(eval_envs.single_action_space.shape).to('cuda')).item()
log_alpha = torch.zeros(1, requires_grad=True, device='cuda')
alpha = log_alpha.exp().item()
a_optimizer = optim.Adam([log_alpha], lr=3e-4)

actor sd = 29
q_net sd = 29
q_net sd = 29
q_net sd = 29
q_net sd = 29


# BC Update

In [47]:
print(batch[0].shape,'\n', 
      batch[1].shape, '\n',
      batch[2].shape, '\n',
      batch[3].shape, '\n',
      batch[4].shape, '\n',
      batch[5].shape, '\n',
      batch[6].shape, '\n',
      batch[7].shape, '\n',
      batch[8].shape, '\n',
      batch[9].shape, '\n',)

torch.Size([60, 5, 29]) 
 torch.Size([60, 5, 64, 64, 3]) 
 torch.Size([60, 5, 29]) 
 torch.Size([60, 5, 64, 64, 3]) 
 torch.Size([60, 5, 4]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 
 torch.Size([60, 5]) 



In [81]:
st_s =  batch[0]
st_i = batch[1]
n_st_s =     batch[2]
n_st_i =    batch[3]
acts =     batch[4]
rew =    batch[5]
dones =    batch[6]
R =     batch[7]
Q=     batch[8]
V=    batch[9]

# Trans Actor (BC)

In [112]:
pred_action = trans_actor.get_action({'state':st_s,'rgb':st_i})[0]

tgt = acts[:,-1:].squeeze(1)

criterion = nn.MSELoss()
trans_actor_loss = criterion(pred_action, tgt)

torch.Size([60, 5, 64, 64, 3])


# Trans Critic (BC)

In [ ]:
q_pred1 = trans_qf1({'state':st_s,'rgb':st_i}, acts[:,-1:].squeeze(1))
q_pred2 = trans_qf2({'state':st_s,'rgb':st_i}, acts[:,-1:].squeeze(1))

tgt1 = qf1({'state':st_s[:,-1,],'rgb':st_i[:,-1,]}, acts[:,-1:].squeeze(1))
tgt2 =  qf2({'state':st_s[:,-1,],'rgb':st_i[:,-1,]}, acts[:,-1:].squeeze(1))

criterion1 = nn.MSELoss()
criterion2 = nn.MSELoss()
trans_qf1_loss = criterion1(q_pred1, tgt1)
trans_qf2_loss = criterion2(q_pred2, tgt2)

torch.Size([60, 5, 64, 64, 3])
torch.Size([60, 5, 64, 64, 3])


# Trans Actor (RL)

In [ ]:
pi, log_pi, _, visual_feature = actor.get_action({'state':st_s,'rgb':st_i})
qf1_pi = trans_qf1({'state':st_s,'rgb':st_i}, pi, visual_feature, detach_encoder=True)
qf2_pi = trans_qf2({'state':st_s,'rgb':st_i}, pi, visual_feature, detach_encoder=True)
min_qf_pi = torch.min(qf1_pi, qf2_pi).view(-1)
actor_loss = ((alpha * log_pi) - min_qf_pi).mean()

actor_optimizer.zero_grad()
actor_loss.backward()
actor_optimizer.step()

autotune = True

if autotune:
    with torch.no_grad(): 
        _, log_pi, _, _ =  actor.get_action({'state':st_s,'rgb':st_i})
    alpha_loss = (-log_alpha.exp() * (log_pi + target_entropy)).mean()
    

    a_optimizer.zero_grad()
    alpha_loss.backward()
    a_optimizer.step()
    alpha = log_alpha.exp().item()

# Trans Critic (RL)

In [ ]:
with torch.no_grad():
    next_state_actions, next_state_log_pi, _, visual_feature = actor.get_action({'state':st_s,'rgb':st_i})
    qf1_next_target = trans_qf1_target({'state':st_s,'rgb':st_i}, next_state_actions, visual_feature)
    qf2_next_target = trans_qf2_target({'state':st_s,'rgb':st_i}, next_state_actions, visual_feature)
    
    next_q_value = Q
    # data.dones is "stop_bootstrap", which is computed earlier according to args.bootstrap_at_done

qf1_a_values = trans_qf1({'state':st_s,'rgb':st_i}, acts[:,-1:].squeeze(1)).view(-1)
qf2_a_values = trans_qf2({'state':st_s,'rgb':st_i}, acts[:,-1:].squeeze(1)).view(-1)
qf1_loss = F.mse_loss(qf1_a_values, Q)
qf2_loss = F.mse_loss(qf2_a_values, Q)
qf_loss = qf1_loss + qf2_loss

q_optimizer.zero_grad()
qf_loss.backward()
q_optimizer.step()